<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [2]:
type(docs)

list

In [5]:
docs[13]

'\n   {Description of "External Tank" option for SSF redesign deleted}\n\n\nYo Ken, let\'s keep on-top of things! Both the "External Tank" and\n"Wingless Orbiter" options have been deleted from the SSF redesign\noptions list. Today\'s (4/23) edition of the New York Times reports\nthat O\'Connor told the panel that some redesign proposals have\nbeen dropped, such as using the "giant external fuel tanks used\nin launching space shuttles," and building a "station around\nan existing space shuttle with its wings and tail removed."\n\nCurrently, there are three options being considered, as presented\nto the advisory panel meeting yesterday (and as reported in\ntoday\'s Times).\n\nOption "A" - Low Cost Modular Approach\nThis option is being studied by a team from MSFC. {As an aside,\nthere are SSF redesign teams at MSFC, JSC, and LaRC supporting\nthe SRT (Station Redesign Team) in Crystal City. Both LeRC and\nReston folks are also on-site at these locations, helping the respective\nteams wit

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [6]:
import re

def preprocess(text):
    # 
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    words = text.split()
    
    words = [w for w in words if len(w) > 2]
    
    return words

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [8]:
def build_vocab(docs):
    vocab = set()
    
    for doc in docs:
        vocab.update(preprocess(doc))
    
    vocab = sorted(list(vocab))
    
    vocab_index = {word: i for i, word in enumerate(vocab)}
    
    return vocab_index

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [9]:
import numpy as np

def compute_df(docs, vocab):
    vocab_index = {w: i for i, w in enumerate(vocab)}
    df = np.zeros(len(vocab))
    
    for doc in docs:
        words = set(preprocess(doc)) 
        for w in words:
            if w in vocab_index:
                df[vocab_index[w]] += 1
                
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [10]:
def compute_idf(df, N):
    return np.log(N / (df + 1)) + 1

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [11]:
def compute_tf(doc, vocab):
    vocab_index = {w: i for i, w in enumerate(vocab)}
    tf = np.zeros(len(vocab))
    
    words = preprocess(doc)
    total_words = len(words)
    
    if total_words == 0:
        return tf
    
    for w in words:
        if w in vocab_index:
            tf[vocab_index[w]] += 1
    
    tf = tf / total_words
    return tf

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [17]:
def build_tfidf(docs):
    vocab = build_vocab(docs)
    vocab_index = {w: i for i, w in enumerate(vocab)}
    
    N = len(docs)
    
    # DF
    df = compute_df(docs, vocab)
    
    # IDF
    idf = compute_idf(df, N)
    tfidf_matrix = []
    
    for doc in docs:
        tf = compute_tf(doc, vocab)
        tfidf = tf * idf
        tfidf_matrix.append(tfidf)
    
    return np.array(tfidf_matrix), vocab, idf

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [13]:
def cosine(a, b):
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return dot / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [18]:
import numpy as np

def search(query, docs, top_k=5):
    tfidf_matrix, vocab, idf = build_tfidf(docs)
    vocab_index = {w: i for i, w in enumerate(vocab)}
    q_vec = np.zeros(len(vocab))

    q_words = preprocess(query)

    for w in q_words:
        if w in vocab_index:
            q_vec[vocab_index[w]] += 1

    q_vec = q_vec / len(q_words)
    q_vec = q_vec * idf

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True, key=lambda x: x[0])

    return scores[:top_k]

> TEST

In [20]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:2])
    print("-" * 50)

0.14815795975674104
I 
--------------------------------------------------
0.147603432767573
 .
--------------------------------------------------
0.13685196927446502
OK
--------------------------------------------------
0.13119948541876642
I'
--------------------------------------------------
0.1297519172249455
[F
--------------------------------------------------


In [21]:
results

[(np.float64(0.14815795975674104), 726),
 (np.float64(0.147603432767573), 1540),
 (np.float64(0.13685196927446502), 430),
 (np.float64(0.13119948541876642), 737),
 (np.float64(0.1297519172249455), 1310)]